In [1]:
import glob
import sys
sys.path.append('/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/LlavaGuard')
import nest_asyncio
nest_asyncio.apply()
import pandas as pd
import os
import json
import sglang as sgl
from llavaguard.taxonomy.PEGI.PEGI_Graph import remove_numbers_from_categories, get_policy_assessment, policy_graph, policy_graph_to_safety_policy_v2, get_rating
from llavaguard.taxonomy.PEGI.PEGI_Graph import get_content_categories, get_content_categories_with_examples, get_content_categories_with_numbers, get_policy_intro, get_safety_categories, policy_graph, policy_graph_to_safety_policy
local_data_dir = '/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo'

In [2]:
#extract category assessments from majority vote
def extract_categories():
    data = f'/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/annotations/auto_generated_annotations/v0_2/majority_vote.csv'
    output_dir = '/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/annotations/auto_generated_annotations/v0_2/'
    df = pd.read_csv(data)
    df = df[df['voting_mechanism'] == 'majority_vote']
    categories = df.columns[3:]
    extracted_data = []
    for index, row in df.iterrows():
        #print('index = ', index)
        for category in categories:
            if row[category] == 1:
                extracted_data.append({
                    'image_name': row['sample_id'],
                    'file_path': row['im_path'],
                    'category': category,
                })
    result_df = pd.DataFrame(extracted_data)
    result_df.to_csv(f'{output_dir}/extracted_categories.csv', index=False)

In [ ]:
#change image paths to correct ones

import pandas as pd

dir = '/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/annotations/auto_generated_annotations/v0_2/'

df = pd.read_csv(f'{dir}/extracted_categories.csv', header=None, names=["image_id", "path", "category"])

old_prefix = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/lhelff/ds/LlavaGuard/data/"
new_prefix = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/"


df["path"] = df["path"].str.replace(old_prefix, new_prefix, regex=False)

df.to_csv(f'{dir}/extracted_categories.csv', index=False, header=False)



In [ ]:
#returns policy, designed for the specific PEGI-Rating of this category

def get_prompt(category):
    return policy_graph_to_safety_policy(get_rating(category))
    

In [ ]:
from transformers import AutoProcessor, Llama4ForConditionalGeneration, AutoTokenizer
import torch
from PIL import Image


model_id = "meta-llama/Llama-4-Scout-17B-16E-Instruct"

processor = AutoProcessor.from_pretrained(model_id)

model = Llama4ForConditionalGeneration.from_pretrained(
    model_id,
    attn_implementation="flex_attention",
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

In [ ]:
#testing llama4 with transformers, only text input
import time

#t0 = time.perf_counter()
import transformers
#print("Import transformers took", time.perf_counter() - t0, "seconds")

#t0 = time.perf_counter()
from transformers import AutoTokenizer
#print("Import AutoTokenizer took", time.perf_counter() - t0, "seconds")

#t0 = time.perf_counter()
from transformers import AutoProcessor

#t0 = time.perf_counter()
from transformers import Llama4ForConditionalGeneration
#print("Import Llama4ForConditionalGeneration took", time.perf_counter() - t0, "seconds")

####

model_id = "meta-llama/Llama-4-Scout-17B-16E-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
)

model = Llama4ForConditionalGeneration.from_pretrained(
    model_id, device_map="auto", 
    torch_dtype=torch.bfloat16,
)

outputs = model.generate(**inputs.to(model.device), max_new_tokens=100)
outputs = tokenizer.batch_decode(outputs[:, inputs["input_ids"].shape[-1] :])
print(outputs[0])

In [ ]:
import os
import argparse
import glob
import sys
sys.path.append('/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/LlavaGuard')
import torch
print("before transformers")
from transformers import AutoProcessor, Llama4ForConditionalGeneration
print("after transformers")
from PIL import Image
from llavaguard_config import local_image_dirs, local_data_dir

import pandas as pd
import json

#loading model 

model_id = "meta-llama/Llama-4-Scout-17B-16E-Instruct"

processor = AutoProcessor.from_pretrained(model_id)

model = Llama4ForConditionalGeneration.from_pretrained(
    model_id,
    attn_implementation="flex_attention",
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

#generating rationales for all the images 


processor = AutoProcessor.from_pretrained(model_id)
local_data_dir = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo" 
csv_path      = f"{local_data_dir}/data/annotations/auto_generated_annotations/v0_2/extracted_categories_stripped.csv"
output_path   = f"{local_data_dir}/data/PEGI-LlavaGuard-DS/all_data.json"
model_id      = "meta-llama/Llama-4-Scout-17B-16E-Instruct"


df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} rows from:\n  {csv_path}")
#df = df.tail(5)
df = df.head(5)
total = len(df)
all_outputs = []
checkpoint_every = 2

for idx, row in df.iterrows():
    sample_id = row["image_name"]
    print(row["file_path"])
    img_path  = row["file_path"]
    category  = row["category"]

    prompt = (
        f"Describe the image and how it relates to the following PEGI rating safety-category - {category} - in 2 sentences."
    )
    print(f"\n[Sample {idx + 1} / {len(df)}]")
    #print(img_path)
    image = Image.open(img_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model.generate(**inputs, max_new_tokens=200)

    start = inputs["input_ids"].shape[-1]
    rationale_text = processor.batch_decode(outputs[:, start:])[0].strip()
    #print("  →  Generated rationale:")
    #print(f"    {rationale_text}")

    all_outputs.append({
        "image_name": sample_id,
        "image_path": img_path,
        "category":   category,
        "rationale":  rationale_text
    })
    if (idx + 1) % checkpoint_every == 0 or idx + 1 == total:
        parent_dir = os.path.dirname(output_path)
        os.makedirs(parent_dir, exist_ok=True)
        with open(output_path, "w") as f:
            json.dump(all_outputs, f, indent=4)
        print(f"  → [Checkpoint] Wrote {idx+1} / {total} entries to {output_path}", flush=True)

#parent_dir = os.path.dirname(output_path)
#os.makedirs(parent_dir, exist_ok=True)
#
#with open(output_path, "w") as f:
#    json.dump(all_outputs, f, indent=4)

print(f"\n Written {len(all_outputs)} entries to:\n {output_path}")

    

In [8]:
from llavaguard_config import local_image_dirs, local_data_dir
print(local_data_dir)

/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo
